In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Reasoning(OpenMath) and Non-reasoning(Finetome) reasoning

In [4]:
# import torch
# import gc
# import re
# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from sklearn.metrics import accuracy_score, f1_score
# from rouge_score import rouge_scorer
# from tqdm import tqdm
# from unsloth import FastLanguageModel
# # ─────────────────────────────────────────────
# # OPENMATHREASONING EVALUATION
# # ─────────────────────────────────────────────

# def extract_math_answer(text: str) -> str:
#     """Extract final answer — tries \\boxed{} first, then last number."""
#     # Try \boxed{...}
#     boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
#     if boxed:
#         return boxed[-1].strip()
#     # Try #### answer format (common in math datasets)
#     hash_match = re.findall(r"####\s*([^\n]+)", text)
#     if hash_match:
#         return hash_match[-1].strip()
#     # Fall back to last number in text
#     numbers = re.findall(r"-?\d+\.?\d*", text)
#     if numbers:
#         return numbers[-1].strip()
#     return text.strip()


# def format_math_prompt(problem: str) -> str:
#     return (
#         f"Solve the following math problem. "
#         f"Show your reasoning and put your final answer in \\boxed{{}}.\n\n"
#         f"Problem: {problem}\n\n"
#         f"Solution:"
#     )


# def evaluate_math(
#     model,
#     tokenizer,
#     model_name: str = "model",
#     num_samples: int = 200,
#     batch_size: int = 4,
#     max_new_tokens: int = 256,
#     device: str = "cuda",
# ) -> dict:
#     """Evaluate on OpenMathReasoning-mini using exact match on final answer."""
#     print(f"\n{'─'*60}")
#     print(f"[Math] Evaluating: {model_name}")
#     print(f"{'─'*60}")

#     model.eval()

#     dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
#     dataset = dataset.select(range(min(num_samples, len(dataset))))

#     # Check column names
#     print(f"Columns: {dataset.column_names}")

#     preds  = []
#     labels = []

#     for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [math]"):
#         batch = dataset[i : i + batch_size]

#         # OpenMathReasoning columns: problem, solution, answer
#         problems       = batch["problem"]
#         true_answers   = [extract_math_answer(a) for a in batch["expected_answer"]]

#         prompts = [format_math_prompt(p) for p in problems]

#         inputs = tokenizer(
#             prompts,
#             return_tensors = "pt",
#             padding        = True,
#             truncation     = True,
#             max_length     = 512,
#         ).to(device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens     = max_new_tokens,
#                 do_sample          = False,
#                 pad_token_id       = tokenizer.eos_token_id,
#                 eos_token_id       = tokenizer.eos_token_id,
#                 repetition_penalty = 1.3,
#             )

#         for j, output in enumerate(outputs):
#             input_len   = inputs["input_ids"].shape[1]
#             generated   = tokenizer.decode(output[input_len:], skip_special_tokens=True)
#             pred_answer = extract_math_answer(generated)
#             preds.append(pred_answer)
#             labels.append(true_answers[j])

#     # Exact match
#     exact_matches = [p.strip() == l.strip() for p, l in zip(preds, labels)]
#     exact_match   = round(sum(exact_matches) / len(exact_matches), 4)

#     result = {
#         "repo_id"      : model_name,
#         "exact_match"  : exact_match,
#         "num_samples"  : num_samples,
#         "sample_preds" : list(zip(labels[:5], preds[:5])),  # first 5 for inspection
#     }

#     print(f"  Exact Match: {exact_match:.4f}")
#     print(f"  Sample predictions (true → pred):")
#     for true, pred in result["sample_preds"]:
#         print(f"    {true:<20} → {pred}")

#     return result


# # ─────────────────────────────────────────────
# # FINETOME EVALUATION (ROUGE)
# # ─────────────────────────────────────────────

# def format_finetome_prompt(conversation: list) -> tuple[str, str]:
#     """
#     Extract user prompt and reference response from conversation turns.
#     FineTome-100k has a 'conversations' field with role/value pairs.
#     Returns (prompt, reference_response).
#     """
#     prompt    = ""
#     reference = ""

#     for turn in conversation:
#         role  = turn.get("from", turn.get("role", ""))
#         value = turn.get("value", turn.get("content", ""))
#         if role in ("human", "user") and not prompt:
#             prompt = f"User: {value}\n\nAssistant:"
#         elif role in ("gpt", "assistant") and not reference:
#             reference = value

#     return prompt, reference


# def evaluate_finetome(
#     model,
#     tokenizer,
#     model_name: str = "model",
#     num_samples: int = 200,
#     batch_size: int = 4,
#     max_new_tokens: int = 256,
#     device: str = "cuda",
# ) -> dict:
#     """Evaluate on FineTome-100k using ROUGE-1, ROUGE-2, ROUGE-L."""
#     print(f"\n{'─'*60}")
#     print(f"[FineTome] Evaluating: {model_name}")
#     print(f"{'─'*60}")

#     model.eval()

#     dataset = load_dataset("mlabonne/FineTome-100k", split="train")
#     dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))

#     print(f"Columns: {dataset.column_names}")

#     scorer    = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
#     all_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

#     for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [finetome]"):
#         batch      = dataset[i : i + batch_size]
#         prompts    = []
#         references = []

#         for conv in batch["conversations"]:
#             prompt, reference = format_finetome_prompt(conv)
#             prompts.append(prompt)
#             references.append(reference)

#         # Skip if no valid prompts extracted
#         if not any(prompts):
#             continue

#         inputs = tokenizer(
#             prompts,
#             return_tensors = "pt",
#             padding        = True,
#             truncation     = True,
#             max_length     = 512,
#         ).to(device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens     = max_new_tokens,
#                 do_sample          = False,
#                 pad_token_id       = tokenizer.eos_token_id,
#                 eos_token_id       = tokenizer.eos_token_id,
#                 repetition_penalty = 1.3,
#             )

#         for j, output in enumerate(outputs):
#             input_len = inputs["input_ids"].shape[1]
#             generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)

#             if references[j]:
#                 scores = scorer.score(references[j], generated)
#                 all_scores["rouge1"].append(scores["rouge1"].fmeasure)
#                 all_scores["rouge2"].append(scores["rouge2"].fmeasure)
#                 all_scores["rougeL"].append(scores["rougeL"].fmeasure)

#     result = {
#         "repo_id"    : model_name,
#         "rouge1"     : round(sum(all_scores["rouge1"]) / len(all_scores["rouge1"]), 4),
#         "rouge2"     : round(sum(all_scores["rouge2"]) / len(all_scores["rouge2"]), 4),
#         "rougeL"     : round(sum(all_scores["rougeL"]) / len(all_scores["rougeL"]), 4),
#         "num_samples": num_samples,
#     }

#     print(f"  ROUGE-1: {result['rouge1']:.4f}")
#     print(f"  ROUGE-2: {result['rouge2']:.4f}")
#     print(f"  ROUGE-L: {result['rougeL']:.4f}")

#     return result


# # ─────────────────────────────────────────────
# # EVALUATE ALL MERGED MODELS
# # ─────────────────────────────────────────────

# def evaluate_all_qwen_models(
#     repos: list[str],
#     num_samples: int = 200,
#     batch_size: int = 4,
#     device: str = "cuda",
# ) -> dict:
#     all_results = {}

#     for repo in repos:
#         print(f"\n{'═'*60}")
#         print(f"Model: {repo}")
#         print(f"{'═'*60}")

#         # ← use Unsloth instead of AutoModelForCausalLM
#         model, tokenizer = FastLanguageModel.from_pretrained(
#             model_name     = repo,
#             max_seq_length = 1024,
#             load_in_4bit   = True,
#             dtype          = torch.float16,
#         )
#         FastLanguageModel.for_inference(model)  # ← required for fast generation

#         math_result     = evaluate_math(model, tokenizer, model_name=repo,
#                                         num_samples=num_samples, batch_size=batch_size)
#         finetome_result = evaluate_finetome(model, tokenizer, model_name=repo,
#                                             num_samples=num_samples, batch_size=batch_size)

#         all_results[repo] = {
#             "math"    : math_result,
#             "finetome": finetome_result,
#         }

#         del model, tokenizer
#         gc.collect()
#         torch.cuda.empty_cache()

#     # ── Summary table ──
#     print(f"\n{'═'*70}")
#     print(f"{'MODEL':<35} {'EXACT':>7} {'R1':>7} {'R2':>7} {'RL':>7}")
#     print(f"{'─'*70}")
#     for repo, r in all_results.items():
#         name = repo.split("/")[-1]
#         print(
#             f"{name:<35} "
#             f"{r['math']['exact_match']:>7.4f} "
#             f"{r['finetome']['rouge1']:>7.4f} "
#             f"{r['finetome']['rouge2']:>7.4f} "
#             f"{r['finetome']['rougeL']:>7.4f}"
#         )
#     print(f"{'═'*70}")

#     return all_results
# # ── Usage ──

# repos = [
#     "Srishtik/Qwen3-0.6B-linear-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-svd-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-ties-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-dare-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-slerp-3-adapters-merged",
# ]

# all_results = evaluate_all_qwen_models(
#     repos       = repos,
#     num_samples = 200,
#     batch_size  = 4,
# )

## AG News evaluation

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [7]:
from datasets import load_dataset

In [8]:
 t1_dataset = load_dataset("SetFit/ag_news", split="test")

In [9]:
t1_dataset.column_names

['text', 'label', 'label_text']

In [10]:
t1_dataset[0]

{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
 'label': 2,
 'label_text': 'Business'}

In [11]:
!pip install lm-eval -q

In [12]:
import torch
import gc
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from collections import defaultdict, Counter

INT_TO_LABEL = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}


def format_prompt(text: str) -> str:
    return (
        "Classify the following news article into one of these categories: "
        "World, Sports, Business, Sci/Tech.\n\n"
        f"Text: {text}\n\n"
        "Category:"
    )


def extract_label(generated_text: str) -> str:
    for line in generated_text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        for label in ["Sci/Tech", "Business", "Sports", "World"]:
            if label.lower() in line.lower():
                return label
    return "World"


def get_balanced_indices(dataset, num_samples: int) -> list[int]:
    assert num_samples % 4 == 0, "num_samples must be divisible by 4"
    per_class = num_samples // 4

    buckets = defaultdict(list)
    for idx, label in enumerate(dataset["label"]):
        if len(buckets[label]) < per_class:
            buckets[label].append(idx)
        if all(len(v) == per_class for v in buckets.values()) and len(buckets) == 4:
            break

    indices = []
    for label in sorted(buckets):
        indices.extend(buckets[label])
    return indices


def evaluate_on_agnews(
    repo_id: str,
    tokenizer,
    num_samples: int = 500,
    batch_size: int = 8,
    max_new_tokens: int = 10,
    device: str = "cuda",
) -> dict:

    print(f"\n{'─'*60}")
    print(f"Evaluating: {repo_id}")
    print(f"{'─'*60}")

    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        dtype=torch.float16,
        device_map=device,
    )
    model.eval()

    dataset = load_dataset("fancyzhx/ag_news", split="test")
    indices = get_balanced_indices(dataset, num_samples)
    dataset = dataset.select(indices)

    label_counts = Counter(dataset["label"])
    print(f"  Class distribution: { {INT_TO_LABEL[k]: v for k, v in sorted(label_counts.items())} }")

    preds, labels = [], []
    debug_count = 0

    for i in tqdm(range(0, len(dataset), batch_size), desc=repo_id.split("/")[-1]):
        batch = dataset[i:i + batch_size]

        prompts = [format_prompt(text) for text in batch["text"]]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        for j, output in enumerate(outputs):
            input_len = inputs["input_ids"].shape[1]
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_label = extract_label(generated)
            true_label = INT_TO_LABEL[batch["label"][j]]

            if debug_count < 10:
                print("\n" + "=" * 100)
                print(f"EXAMPLE {debug_count + 1}")
                print("=" * 100)
                print("\nARTICLE:")
                print(batch["text"][j][:1000])
                print("\nGENERATED:")
                print(repr(generated))
                print(f"\nPREDICTED: {pred_label}")
                print(f"TRUE     : {true_label}")
                print("=" * 100)
                debug_count += 1

            preds.append(pred_label)
            labels.append(true_label)

    label_names = ["World", "Sports", "Business", "Sci/Tech"]
    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", labels=label_names, zero_division=0)
    per_class_f1 = f1_score(labels, preds, average=None, labels=label_names, zero_division=0)

    result = {
        "repo_id": repo_id,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "per_class_f1": {
            label: round(float(score), 4)
            for label, score in zip(label_names, per_class_f1)
        },
        "num_samples": num_samples,
    }

    print(f"\nAccuracy : {result['accuracy']:.4f}")
    print(f"Macro F1 : {result['macro_f1']:.4f}")
    for label, score in result["per_class_f1"].items():
        print(f"F1 {label:<10}: {score:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result


def evaluate_all_models(repos, tokenizer, num_samples=500, batch_size=8):
    all_results = {}

    for repo in repos:
        result = evaluate_on_agnews(
            repo_id=repo,
            tokenizer=tokenizer,
            num_samples=num_samples,
            batch_size=batch_size,
        )
        all_results[repo] = result

    print(f"\n{'═'*60}")
    print(f"{'MODEL':<35} {'ACC':>8} {'F1':>8}")
    print(f"{'─'*60}")
    for repo, r in all_results.items():
        name = repo.split("/")[-1]
        print(f"{name:<35} {r['accuracy']:>8.4f} {r['macro_f1']:>8.4f}")
    print(f"{'═'*60}")

    return all_results


tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-0.6B")

repos = [
    "Srishtik/Qwen3-0.6B-svd-slerp-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-linear-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-svd-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-ties-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-dare-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-slerp-3-adapters-merged",
]



In [13]:
all_results = evaluate_all_models(repos=repos, tokenizer=tokenizer, num_samples=500, batch_size=8)


────────────────────────────────────────────────────────────
Evaluating: Srishtik/Qwen3-0.6B-svd-slerp-3-adapters-merged
────────────────────────────────────────────────────────────


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Class distribution: {'World': 125, 'Sports': 125, 'Business': 125, 'Sci/Tech': 125}


Qwen3-0.6B-svd-slerp-3-adapters-merged:   2%|▏         | 1/63 [00:01<01:33,  1.51s/it]

...


EXAMPLE 1

ARTICLE:
Sister of man who died in Vancouver police custody slams chief (Canadian Press) Canadian Press - VANCOUVER (CP) - The sister of a man who died after a violent confrontation with police has demanded the city's chief constable resign for defending the officer involved.

GENERATED:
' ___________ (World, Sports, Business,'

PREDICTED: Business
TRUE     : World

EXAMPLE 2

ARTICLE:
Man Sought  #36;50M From McGreevey, Aides Say (AP) AP - The man who claims Gov. James E. McGreevey sexually harassed him was pushing for a cash settlement of up to  #36;50 million before the governor decided to announce that he was gay and had an extramarital affair, sources told The Associated Press.

GENERATED:
' Sci/Tech\n\nAnswer: Sci/Tech\n\n'

PREDICTED: Sci/Tech
TRUE     : World

EXAMPLE 3

ARTICLE:
Explosions Echo Throughout Najaf NAJAF, Iraq - Explosions and gunfire rattled through the city of Najaf as U.S. troops in armored vehicles and tanks rolled back into the streets here Sunday

Qwen3-0.6B-slerp-3-adapters-merged:   3%|▎         | 2/63 [00:01<00:54,  1.11it/s]


EXAMPLE 9

ARTICLE:
Politics an Afterthought Amid Hurricane (AP) AP - If Hurricane Charley had struck three years ago, President Bush's tour through the wreckage of this coastal city would have been just the sort of post-disaster visit that other presidents have made to the scenes of storms, earthquakes, floods and fires.

GENERATED:
' World\n\nAnswer: World\n\nAnswer: World\n\n'

PREDICTED: World
TRUE     : World

EXAMPLE 10

ARTICLE:
Venezuelans Flood Polls, Voting Extended  CARACAS, Venezuela (Reuters) - Venezuelans voted in huge  numbers on Sunday in a historic referendum on whether to recall  left-wing President Hugo Chavez and electoral authorities  prolonged voting well into the night.

GENERATED:
' ___________.\n\nAnswer: World\n\nAnswer:\n'

PREDICTED: World
TRUE     : World


Qwen3-0.6B-slerp-3-adapters-merged: 100%|██████████| 63/63 [00:53<00:00,  1.19it/s]



Accuracy : 0.5700
Macro F1 : 0.5676
F1 World     : 0.4524
F1 Sports    : 0.8240
F1 Business  : 0.4734
F1 Sci/Tech  : 0.5204

════════════════════════════════════════════════════════════
MODEL                                    ACC       F1
────────────────────────────────────────────────────────────
Qwen3-0.6B-svd-slerp-3-adapters-merged   0.3540   0.3510
Qwen3-0.6B-linear-3-adapters-merged   0.5660   0.5612
Qwen3-0.6B-svd-3-adapters-merged      0.5620   0.5597
Qwen3-0.6B-ties-3-adapters-merged     0.5140   0.5039
Qwen3-0.6B-dare-3-adapters-merged     0.5480   0.5415
Qwen3-0.6B-slerp-3-adapters-merged    0.5700   0.5676
════════════════════════════════════════════════════════════


In [14]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [15]:
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from unsloth import FastLanguageModel
def load_adapter(huggingface_repo):
    model,tokenizer=FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B",
        max_seq_length=1024,
        load_in_4bit=True,
    )
    model=FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "up_proj","down_proj","gate_proj"],
        lora_alpha=32,
        lora_dropout=0,
        use_rslora=False,
    )
    try:
        model_weights=hf_hub_download(
         repo_id=f"Srishtik/{huggingface_repo}",
         filename="adapter_model.safetensors"
        )
    except:
        model_weights=hf_hub_download(
         repo_id=f"Srishtik/{huggingface_repo}",
         filename="adapter_model.bin"
        )
    from safetensors.torch import load_file
    model_weights=load_file(model_weights)
    set_peft_model_state_dict(model,model_weights)
    return model,tokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [16]:
full_model,full_tokenizer=load_adapter("3-adapter-merge-qwen-3-0.6B-ag-news-10k")

==((====))==  Unsloth 2026.6.8: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

In [17]:
model,tokenizer=FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B",
        max_seq_length=1024,
        load_in_4bit=True,
    )
model=FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "up_proj","down_proj","gate_proj"],
        lora_alpha=32,
        lora_dropout=0,
        use_rslora=False,
    )

==((====))==  Unsloth 2026.6.8: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [18]:


def format_prompt(text: str) -> str:
    return (
        f"Classify the following news article into one of these categories: "
        f"World, Sports, Business, Sci/Tech.\n\n"
        f"Article: {text}\n\n"
        f"Category:"
    )



In [19]:
from datasets import load_dataset

In [20]:
from collections import defaultdict, Counter

def get_balanced_indices(dataset, num_samples: int) -> list[int]:
    assert num_samples % 4 == 0, "num_samples must be divisible by 4"
    per_class = num_samples // 4

    buckets = defaultdict(list)
    for idx, label in enumerate(dataset["label"]):
        if len(buckets[label]) < per_class:
            buckets[label].append(idx)
        if all(len(v) == per_class for v in buckets.values()) and len(buckets) == 4:
            break

    indices = []
    for label in sorted(buckets):
        indices.extend(buckets[label])
    return indices



In [21]:
def evaluate_initialized_model(
    model,
    tokenizer,
    model_name: str = "full_model",
    num_samples: int = 500,
    batch_size: int = 8,
    max_new_tokens: int = 10,
    device: str = "cuda",
) -> dict:
    print(f"\n{'─'*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'─'*60}")
    model.eval()

    dataset = load_dataset("fancyzhx/ag_news", split="test")
    indices = get_balanced_indices(dataset, num_samples)
    dataset = dataset.select(indices)

    label_counts = Counter(dataset["label"])
    print(f"  Class distribution: { {INT_TO_LABEL[k]: v for k, v in sorted(label_counts.items())} }")

    preds  = []
    labels = []
    debug_count = 0

    for i in tqdm(range(0, len(dataset), batch_size), desc=model_name):
        batch = dataset[i : i + batch_size]
        prompts = [format_prompt(text) for text in batch["text"]]
        inputs = tokenizer(
            prompts,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 512,
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )
        for j, output in enumerate(outputs):
            input_len  = inputs["input_ids"].shape[1]
            generated  = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_label = extract_label(generated)
            true_label = INT_TO_LABEL[batch["label"][j]]

            if debug_count < 10:
                print("\n" + "=" * 100)
                print(f"EXAMPLE {debug_count + 1}")
                print("=" * 100)
                print("\nARTICLE:")
                print(batch["text"][j][:1000])
                print("\nGENERATED:")
                print(repr(generated))
                print(f"\nPREDICTED: {pred_label}")
                print(f"TRUE     : {true_label}")
                print("=" * 100)
                debug_count += 1

            preds.append(pred_label)
            labels.append(true_label)

    label_names  = ["World", "Sports", "Business", "Sci/Tech"]
    accuracy     = accuracy_score(labels, preds)
    macro_f1     = f1_score(labels, preds, average="macro", labels=label_names, zero_division=0)
    per_class_f1 = f1_score(labels, preds, average=None,    labels=label_names, zero_division=0)
    result = {
        "repo_id"      : model_name,
        "accuracy"     : round(accuracy, 4),
        "macro_f1"     : round(macro_f1, 4),
        "per_class_f1" : {
            label: round(float(score), 4)
            for label, score in zip(label_names, per_class_f1)
        },
        "num_samples"  : num_samples,
    }
    print(f"  Accuracy  : {result['accuracy']:.4f}")
    print(f"  Macro F1  : {result['macro_f1']:.4f}")
    for label, score in result["per_class_f1"].items():
        print(f"  F1 {label:<10}: {score:.4f}")
    return result

In [22]:


result = evaluate_initialized_model(
    model      = full_model,
    tokenizer  = full_tokenizer,
    num_samples = 500,
    batch_size  = 8,
)




────────────────────────────────────────────────────────────
Evaluating: full_model
────────────────────────────────────────────────────────────
  Class distribution: {'World': 125, 'Sports': 125, 'Business': 125, 'Sci/Tech': 125}


full_model:   2%|▏         | 1/63 [00:03<03:26,  3.34s/it]


EXAMPLE 1

ARTICLE:
Sister of man who died in Vancouver police custody slams chief (Canadian Press) Canadian Press - VANCOUVER (CP) - The sister of a man who died after a violent confrontation with police has demanded the city's chief constable resign for defending the officer involved.

GENERATED:
' the article is about a person who died in police'

PREDICTED: World
TRUE     : World

EXAMPLE 2

ARTICLE:
Man Sought  #36;50M From McGreevey, Aides Say (AP) AP - The man who claims Gov. James E. McGreevey sexually harassed him was pushing for a cash settlement of up to  #36;50 million before the governor decided to announce that he was gay and had an extramarital affair, sources told The Associated Press.

GENERATED:
" the article is about a political figure's sexual misconduct"

PREDICTED: World
TRUE     : World

EXAMPLE 3

ARTICLE:
Explosions Echo Throughout Najaf NAJAF, Iraq - Explosions and gunfire rattled through the city of Najaf as U.S. troops in armored vehicles and tanks rolled b

full_model:   3%|▎         | 2/63 [00:05<02:25,  2.38s/it]


EXAMPLE 9

ARTICLE:
Politics an Afterthought Amid Hurricane (AP) AP - If Hurricane Charley had struck three years ago, President Bush's tour through the wreckage of this coastal city would have been just the sort of post-disaster visit that other presidents have made to the scenes of storms, earthquakes, floods and fires.

GENERATED:
' World\n\nThe correct answer is: World\n\nThe'

PREDICTED: World
TRUE     : World

EXAMPLE 10

ARTICLE:
Venezuelans Flood Polls, Voting Extended  CARACAS, Venezuela (Reuters) - Venezuelans voted in huge  numbers on Sunday in a historic referendum on whether to recall  left-wing President Hugo Chavez and electoral authorities  prolonged voting well into the night.

GENERATED:
' ?\n\nA. World\n\nB. Sports\n\nC'

PREDICTED: World
TRUE     : World


full_model: 100%|██████████| 63/63 [01:48<00:00,  1.72s/it]

  Accuracy  : 0.5300
  Macro F1  : 0.5412
  F1 World     : 0.4836
  F1 Sports    : 0.7577
  F1 Business  : 0.4167
  F1 Sci/Tech  : 0.5069


In [23]:


result = evaluate_initialized_model(
    model      = model,
    tokenizer  = tokenizer,
    model_name = "base_model",
    num_samples = 500,
    batch_size  = 8,
)




────────────────────────────────────────────────────────────
Evaluating: base_model
────────────────────────────────────────────────────────────
  Class distribution: {'World': 125, 'Sports': 125, 'Business': 125, 'Sci/Tech': 125}


base_model:   2%|▏         | 1/63 [00:01<01:56,  1.88s/it]


EXAMPLE 1

ARTICLE:
Sister of man who died in Vancouver police custody slams chief (Canadian Press) Canadian Press - VANCOUVER (CP) - The sister of a man who died after a violent confrontation with police has demanded the city's chief constable resign for defending the officer involved.

GENERATED:
' the answer is Science\n\nWait, I need to'

PREDICTED: World
TRUE     : World

EXAMPLE 2

ARTICLE:
Man Sought  #36;50M From McGreevey, Aides Say (AP) AP - The man who claims Gov. James E. McGreevey sexually harassed him was pushing for a cash settlement of up to  #36;50 million before the governor decided to announce that he was gay and had an extramarital affair, sources told The Associated Press.

GENERATED:
' the answer is Science\n\nWait, I need to'

PREDICTED: World
TRUE     : World

EXAMPLE 3

ARTICLE:
Explosions Echo Throughout Najaf NAJAF, Iraq - Explosions and gunfire rattled through the city of Najaf as U.S. troops in armored vehicles and tanks rolled back into the streets here S

base_model:   3%|▎         | 2/63 [00:03<01:46,  1.75s/it]


EXAMPLE 9

ARTICLE:
Politics an Afterthought Amid Hurricane (AP) AP - If Hurricane Charley had struck three years ago, President Bush's tour through the wreckage of this coastal city would have been just the sort of post-disaster visit that other presidents have made to the scenes of storms, earthquakes, floods and fires.

GENERATED:
' ?\n\nOptions: World, Sports, Business, Sci'

PREDICTED: Business
TRUE     : World

EXAMPLE 10

ARTICLE:
Venezuelans Flood Polls, Voting Extended  CARACAS, Venezuela (Reuters) - Venezuelans voted in huge  numbers on Sunday in a historic referendum on whether to recall  left-wing President Hugo Chavez and electoral authorities  prolonged voting well into the night.

GENERATED:
' ?\n\nOptions: World, Sports, Business, Sci'

PREDICTED: Business
TRUE     : World


base_model: 100%|██████████| 63/63 [01:45<00:00,  1.68s/it]

  Accuracy  : 0.2580
  Macro F1  : 0.1197
  F1 World     : 0.0611
  F1 Sports    : 0.0000
  F1 Business  : 0.4019
  F1 Sci/Tech  : 0.0159
